# 最优水桶问题

**类别：** 非线性

来源：[https://www.hexaly.com/templates/optimal-bucket-problem](https://www.hexaly.com/templates/optimal-bucket-problem)


## 问题描述

水桶的最佳形状是什么？在 **最优水桶问题** 中，我们希望设计一个能在不超过可用表面材料的情况下最大化所能容纳流体体积的水桶。一个水桶由三个值定义：底部圆盘的半径、顶部开口的半径以及高度，分别记为 r、R 和 h。问题在于选择 r、R 和 h 的值，以在水桶表面积不超过可用材料的约束下，最大化水桶的体积。

更多细节请参见 [DataGenetics](http://datagenetics.com/blog/january32015/index.html)。

	

### 建模要点

- 添加 [浮点决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/operatorsreference.html#floating-point-decisions) 来建模水桶的尺寸
- 使用 [非线性算子](https://www.hexaly.com/docs/last/mathematicaloperators/operatorsreference.html#table-of-available-operators-and-functions) 来计算水桶的表面积和体积
- 了解 OptAgent 的建模风格：[区分决策变量与中间表达式](https://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#distinguish-decision-variables-from-intermediate-variableshttps://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#distinguish-decision-variables-from-intermediate-variables)


## 模型

用于建造水桶的可用材料是一个半径为 1 的平面圆盘，其表面积为 S=π。在不超出该材料面积的前提下，我们尝试构造一个能容纳最大体积的水桶。

模型包含三个浮点决策变量。它们代表定义水桶形状的三个量：底部圆盘半径 r、顶部开口半径 R 以及高度 h。水桶的表面积和体积完全由这三个量确定，因此它们无需作为决策变量，而是作为中间表达式。水桶的表面积由表达式 S = π*r² + π(R+r)sqrt((R-r)²+h²) 给出。在计算之后，我们将其约束为不超过 π。然后我们可以计算水桶的体积，由表达式 V = (π*h)/3 * (R²+Rr+r²) 给出，并将其最大化。


## Python 实现


In [3]:
from pathlib import Path

from optagent import OptModel, solve


def main(output_file=None, time_limit=2):
    pi = 3.14159265359
    model = OptModel()

    # Numerical decisions defining the bucket shape.
    radius_top = model.float(0, 1, name="R")
    radius_bottom = model.float(0, 1, name="r")
    height = model.float(0, 1, name="h")

    # Keep surface area and volume as intermediate nonlinear expressions.
    surface = pi * radius_bottom**2 + pi * (radius_top + radius_bottom) * model.sqrt(
        (radius_top - radius_bottom) ** 2 + height**2
    )
    model.constraint(surface <= pi, name="surface_limit")
    volume = pi * height / 3 * (radius_top**2 + radius_top * radius_bottom + radius_bottom**2)
    model.maximize(volume, name="volume")

    solution = solve(model, time_limit_s=float(time_limit))
    result_text = (
        f"Surface = {surface.value:.6f}; Volume = {volume.value:.6f}; "
        f"R = {radius_top.value:.6f}; r = {radius_bottom.value:.6f}; "
        f"h = {height.value:.6f}; Status = {solution.feasible}"
    )
    print(result_text)
    if output_file is not None:
        Path(output_file).write_text(
            f"{surface.value} {volume.value}\n"
            f"{radius_top.value} {radius_bottom.value} {height.value}\n",
            encoding="utf-8",
        )
    return solution


## 运行实例

以下代码格演示如何调用 OptAgent 的最优水桶模型。

In [4]:
solution = main(time_limit=1)


Starting OptAgent
Parameters: time_limit=1s
[   0.002s] initial feasible=true objective=[0]
[   0.007s] best #1 worker=0 feasible=true objective=[0.065449846949791668]
[   0.213s] best #19 worker=2 feasible=true objective=[0.65478684973651025]
[   0.431s] best #44 worker=2 feasible=true objective=[0.65602487880530602]
[   0.635s] best #78 worker=2 feasible=true objective=[0.65602487880531579]
[   0.839s] best #106 worker=2 feasible=true objective=[0.65602487880532245]
[   1.005s] best #126 worker=2 feasible=true objective=[0.65602487880532678]
Solve summary:
  status: FEASIBLE
  objective: [0.65602487880532678]
  improvements: 126
  evaluated: 4333
  wall_time: 1.00503s
  termination: deadline


Surface = 3.141593; Volume = 0.656025; R = 0.714037; r = 0.145374; h = 0.986884; Status = True
